In [ ]:

# ! pip install -r ../../../../requirements.txt -U

In [ ]:
from agent_framework import Message, Content

from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity.aio import AzureCliCredential

In [ ]:

import os
import base64
from dotenv import load_dotenv


In [ ]:
load_dotenv("../../../../.env")

In [ ]:
AgentName = "Vision-Agent"
AgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
async with (
    AzureCliCredential() as credential,
    AzureAIProjectAgentProvider(project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"], credential=credential) as client,
):
    agent = await client.create_agent(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions=AgentInstructions,
        name=AgentName
    )

    image_path = "../../files/home.png"
    with open(image_path, "rb") as image_file:
        image_b64 = base64.b64encode(image_file.read()).decode()
    image_uri = f"data:image/png;base64,{image_b64}"
    message = Message(
        role="user",
        contents=[
            Content.from_text(text="Please find the relevant furniture according to the image and give the corresponding price for each piece of furniture"),
            Content.from_uri(uri=image_uri, media_type="image/png"),
        ]
    )
    first_result = await agent.run(message)

    print(f"Agent: {first_result.text}")